# Dijet pointing-resolution systematic
Estimate the one-sided pointing uncertainty `abs(Reco/Ref - 1)` in the CM and Lab frames. Full shapes are unit-normalized. The CM workflow also folds F/B from each unnormalized projection using eta > 0.001 as Forward and reflected eta < -0.001 as Backward, with independent errors; no Lab F/B is constructed.

In [ ]:
%load_ext autoreload
%autoreload 2
from dataclasses import replace
from pathlib import Path
import os
import sys

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / 'CMakeLists.txt').is_file()
        and (candidate / 'hist_analysis').is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Start Jupyter from the jetAnalysis repository')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from hist_analysis.python.notebook_setup import load_root
ROOT = load_root(batch=True)
from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import (
    DIJET_DELTA_PHI_SELECTION_LABEL,
    STANDARD_DIJET_ETA_CUT_INDEX,
    TEST_DIJET_PTAVE_BINS,
    DIJET_PTAVE_BINS,
)
from hist_analysis.python.data_distributions import (
    forward_backward_from_full, project_data_histogram,
)
from hist_analysis.python.histogram_io import (
    resolve_combined_file, resolve_direction_file,
)
from hist_analysis.python.histogram_ops import (
    normalize_histogram, ratio_to_nominal,
)
from hist_analysis.python.plotting import draw_overlay
from hist_analysis.python.root_style import (
    COLORS, DEFAULT_PLOT_STYLE, save_canvas,
)
from hist_analysis.python.systematic_fits import (
    calculate_one_sided_systematic,
    fit_histogram_variations,
    format_fit_summary_lines,
    smooth_systematic_running_max,
    write_systematic_csv,
)

ROOT.gStyle.SetOptStat(0)
ROOT.TH1.AddDirectory(False)

## Configuration
Comparison ratios allow standard (`''`) or binomial (`'B'`) errors. `FORWARD_BACKWARD_RATIO_OPTION` is protected: F/B construction is never binomial.

In [ ]:
GENERATOR = 'embedding'  # embedding or pythia
DIRECTION = 'combined'   # combined, Pbgoing, or pgoing
FILE_STEM = 'jetId'
ETA_CUTS = (1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.3, 2.4, 3.0)
# ETA_CUT_INDEX = STANDARD_DIJET_ETA_CUT_INDEX
ETA_CUT_INDEX = 5
LAB_ETA_CUT_INDEX = 6
# PTAVE_BINS = DIJET_PTAVE_BINS
PTAVE_BINS = tuple(TEST_DIJET_PTAVE_BINS)
REBIN_ETA = 2

SYSTEMATIC_EXTRACTION = 'fit'  # fit or bin_by_bin
APPLY_SYSTEMATIC_SMOOTHING = True
FULL_SMOOTHING_ORIGIN = -0.465 + 0.00001

FORWARD_BACKWARD_RATIO_OPTION = ''  # protected: never binomial
FULL_COMPARISON_RATIO_OPTION = 'B'  # '' or 'B'
FB_COMPARISON_RATIO_OPTION = 'B'    # '' or 'B'; after F/B construction

FULL_FIT_FUNCTION = 'pol2'
FB_FIT_FUNCTION = 'pol1'
FULL_FIT_INITIAL_VALUES = {
    'Reco / Ref': (1.0, 0.0, 0.0),
    # 'Reco / Ref': (1.0, 0.0, 0.0, 0.0, 0.0),
}
FB_FIT_INITIAL_VALUES = {
    '(F/B) Reco / Ref': (1.0, 0.0),
}
FIT_OPTIONS = 'RQS0'          # range, result, quiet, no draw
FIT_WEIGHT_OPTION = ''       # 'W': unit weights; '': use bin errors
EFFECTIVE_FIT_OPTIONS = FIT_OPTIONS + FIT_WEIGHT_OPTION
FIT_WEIGHT_TAG = 'weights1' if FIT_WEIGHT_OPTION == 'W' else 'weightsStd'
OUTPUT_CONFIGURATION_TAG = (
    f'fullFit_{FULL_FIT_FUNCTION}_fbFit_{FB_FIT_FUNCTION}'
    f'_{FIT_WEIGHT_TAG}_systCombMax'
)
SHOW_FIT_RESULTS = True

DRAW_GRID = True
SAVE_PNG = False
FULL_RATIO_RANGE = (0.85, 1.15)
FB_RANGE = (0.75, 1.30)
FB_DOUBLE_RATIO_RANGE = (0.85, 1.15)
SYSTEMATIC_Y_RANGE = None
OUTPUT_DIR = Path(os.environ.get(
    'DIJET_POINTING_SYSTEMATICS_OUTPUT_DIR',
    PROJECT_ROOT / 'hist_analysis' / 'output' / 'systematics_pointing_resolution',
))

if GENERATOR not in ('embedding', 'pythia'):
    raise ValueError('GENERATOR must be embedding or pythia')
if DIRECTION not in ('combined', 'Pbgoing', 'pgoing'):
    raise ValueError('DIRECTION must be combined, Pbgoing, or pgoing')
if SYSTEMATIC_EXTRACTION not in ('fit', 'bin_by_bin'):
    raise ValueError("SYSTEMATIC_EXTRACTION must be 'fit' or 'bin_by_bin'")
if FORWARD_BACKWARD_RATIO_OPTION != '':
    raise ValueError('F/B construction must use independent errors')
if FIT_WEIGHT_OPTION not in ('', 'W'):
    raise ValueError("FIT_WEIGHT_OPTION must be empty or 'W'")
for option_name, option in (
    ('FULL_COMPARISON_RATIO_OPTION', FULL_COMPARISON_RATIO_OPTION),
    ('FB_COMPARISON_RATIO_OPTION', FB_COMPARISON_RATIO_OPTION),
):
    if option not in ('', 'B'):
        raise ValueError(f'{option_name} must be empty or B')
for name, index in (('ETA_CUT_INDEX', ETA_CUT_INDEX), ('LAB_ETA_CUT_INDEX', LAB_ETA_CUT_INDEX)):
    if not 0 <= index < len(ETA_CUTS):
        raise IndexError(f'Invalid {name}: {index}')

ETA_CUT = ETA_CUTS[ETA_CUT_INDEX]
LAB_ETA_CUT = ETA_CUTS[LAB_ETA_CUT_INDEX]
if DIRECTION == 'combined':
    INPUT_FILE = resolve_combined_file(BASE_DIR, GENERATOR, FILE_STEM)
else:
    INPUT_FILE = resolve_direction_file(
        BASE_DIR, GENERATOR, DIRECTION, FILE_STEM,
    )
if not INPUT_FILE.exists():
    raise FileNotFoundError(INPUT_FILE)

HISTOGRAM_KEYS = {
    'Reco': f'hRecoDijetPtEtaCM_{ETA_CUT_INDEX}',
    'Ref': f'hRefDijetPtEtaCMPointingRes_{ETA_CUT_INDEX}',
}
LAB_HISTOGRAM_KEYS = {
    'Reco': f'hRecoDijetPtEtaLab_{LAB_ETA_CUT_INDEX}',
    'Ref': f'hRefDijetPtEtaLabPointingRes_{LAB_ETA_CUT_INDEX}',
}
STYLE = replace(
    DEFAULT_PLOT_STYLE,
    annotation_text_size=0.026,
    annotation_line_spacing=0.039,
    legend_text_size=0.028,
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_FILE

In [ ]:
def build_systematic_band(systematic):
    graph = ROOT.TGraphAsymmErrors(systematic.GetNbinsX())
    for bin_index in range(1, systematic.GetNbinsX() + 1):
        uncertainty = systematic.GetBinContent(bin_index)
        half_width = systematic.GetBinWidth(bin_index) / 2.0
        graph.SetPoint(
            bin_index - 1, systematic.GetBinCenter(bin_index), 1.0,
        )
        graph.SetPointError(
            bin_index - 1, half_width, half_width,
            uncertainty, uncertainty,
        )
    graph.SetFillColorAlpha(COLORS[3], 0.30)
    graph.SetLineColor(COLORS[3])
    return graph


def draw_ratio_with_systematic_band(
    ratios, systematic, fit_functions, *, x_range, y_range,
    x_title, y_title, annotations, output, canvas_name,
):
    canvas = draw_overlay(
        ratios, title='', x_title=x_title, y_title=y_title,
        x_range=x_range, y_range=y_range, reference_y=1.0,
        annotations=annotations, grid=DRAW_GRID,
        overlay_functions=fit_functions,
        style_indices={next(iter(ratios)): 0}, style=STYLE,
        output=None, canvas_name=canvas_name,
    )
    graph = build_systematic_band(systematic)
    graph.Draw('E2 SAME')
    for histogram in ratios.values():
        histogram.Draw('E1 SAME')
    for function in fit_functions.values():
        function.Draw('SAME')

    legend = canvas._overlay_objects[0]
    legend.AddEntry(graph, 'Pointing syst. uncrt.', 'f')
    canvas._overlay_objects.append(graph)
    canvas.Modified()
    canvas.Update()
    save_canvas(canvas, output, save_png=SAVE_PNG)
    return canvas

In [ ]:
pointing_results = {}
eta_range = (-ETA_CUT - 0.1, ETA_CUT + 0.1)
fb_range = (0.0, ETA_CUT + 0.1)

for ptave_range in PTAVE_BINS:
    low, high = ptave_range
    ptave_tag = f'{low:g}_{high:g}'.replace('.', 'p')
    tag = (
        f'{GENERATOR}_{DIRECTION}_pointing_etaCM_{round(10 * ETA_CUT):g}'
        f'_ptave_{ptave_tag}'
    )
    output_prefix = f'{GENERATOR}_{DIRECTION}_pointing'
    output_suffix = (
        f'{OUTPUT_CONFIGURATION_TAG}_etaCM_{round(10 * ETA_CUT):g}'
        f'_ptave_{ptave_tag}'
    )

    raw_eta = {
        label: project_data_histogram(
            INPUT_FILE, key, 'eta', ptave_range,
            name=f'h_{tag}_{label.lower()}_raw',
            rebin=REBIN_ETA, normalization='none',
        )
        for label, key in HISTOGRAM_KEYS.items()
    }
    eta_shapes = {
        label: normalize_histogram(histogram, 'integral')
        for label, histogram in raw_eta.items()
    }
    fb_ratios = {
        label: forward_backward_from_full(
            histogram, name=f'h_{tag}_{label.lower()}_fb',
        )
        for label, histogram in raw_eta.items()
    }

    eta_ratios = {
        'Reco / Ref': ratio_to_nominal(
            eta_shapes['Reco'], eta_shapes['Ref'],
            name=f'h_{tag}_reco_ref', option=FULL_COMPARISON_RATIO_OPTION,
        )
    }
    fb_comparisons = {
        '(F/B) Reco / Ref': ratio_to_nominal(
            fb_ratios['Reco'], fb_ratios['Ref'],
            name=f'h_{tag}_fb_reco_ref', option=FB_COMPARISON_RATIO_OPTION,
        )
    }

    eta_fits, eta_fit_summary = fit_histogram_variations(
        eta_ratios, formula=FULL_FIT_FUNCTION,
        fit_range=(-ETA_CUT, ETA_CUT), name_prefix=f'f_{tag}_full',
        fit_options=EFFECTIVE_FIT_OPTIONS,
        initial_values=FULL_FIT_INITIAL_VALUES,
    )
    fb_fits, fb_fit_summary = fit_histogram_variations(
        fb_comparisons, formula=FB_FIT_FUNCTION,
        fit_range=(0.0, ETA_CUT), name_prefix=f'f_{tag}_fb',
        fit_options=EFFECTIVE_FIT_OPTIONS,
        initial_values=FB_FIT_INITIAL_VALUES,
    )

    use_fit = SYSTEMATIC_EXTRACTION == 'fit'
    eta_systematic_unsmoothed = calculate_one_sided_systematic(
        eta_ratios['Reco / Ref'], name=f'h_{tag}_full_syst',
        variation_function=eta_fits['Reco / Ref'] if use_fit else None,
        evaluation_range=(-ETA_CUT, ETA_CUT),
    )
    fb_systematic_unsmoothed = calculate_one_sided_systematic(
        fb_comparisons['(F/B) Reco / Ref'], name=f'h_{tag}_fb_syst',
        variation_function=(
            fb_fits['(F/B) Reco / Ref'] if use_fit else None
        ),
        evaluation_range=(0.0, ETA_CUT),
    )

    eta_systematic = (
        smooth_systematic_running_max(
            eta_systematic_unsmoothed,
            name=f'{eta_systematic_unsmoothed.GetName()}_smoothed',
            evaluation_range=(-ETA_CUT, ETA_CUT),
            smoothing_origin=FULL_SMOOTHING_ORIGIN,
        )
        if APPLY_SYSTEMATIC_SMOOTHING else eta_systematic_unsmoothed
    )
    fb_systematic = (
        smooth_systematic_running_max(
            fb_systematic_unsmoothed,
            name=f'{fb_systematic_unsmoothed.GetName()}_smoothed',
            evaluation_range=(0.0, ETA_CUT),
        )
        if APPLY_SYSTEMATIC_SMOOTHING else fb_systematic_unsmoothed
    )

    eta_percent = eta_systematic.Clone(f'{eta_systematic.GetName()}_percent')
    eta_percent.SetDirectory(0)
    eta_percent.Scale(100.0)
    fb_percent = fb_systematic.Clone(f'{fb_systematic.GetName()}_percent')
    fb_percent.SetDirectory(0)
    fb_percent.Scale(100.0)

    annotations = (
        GENERATOR.capitalize(), DIRECTION,
        f'{low:g} < p_{{T}}^{{ave,reco}} < {high:g} GeV',
        f'|#eta_{{CM}}^{{jet}}| < {ETA_CUT:g}',
        DIJET_DELTA_PHI_SELECTION_LABEL,
    )
    common_plot_options = dict(
        title='', annotations=annotations, grid=DRAW_GRID,
        style=STYLE, save_png=SAVE_PNG,
    )
    canvases = {}
    canvases['eta_overlay'] = draw_overlay(
        eta_shapes, x_title='#eta_{CM}^{dijet}',
        y_title='1/N dN/d#eta_{CM}^{dijet}', x_range=eta_range,
        style_indices={'Reco': 0, 'Ref': 1},
        output=OUTPUT_DIR / f'{output_prefix}_full_overlay_{output_suffix}.pdf',
        canvas_name=f'{tag}_full_overlay', **common_plot_options,
    )
    canvases['eta_ratio'] = draw_overlay(
        eta_ratios, x_title='#eta_{CM}^{dijet}', y_title='Reco / Ref',
        x_range=eta_range, y_range=FULL_RATIO_RANGE, reference_y=1.0,
        overlay_functions=eta_fits,
        overlay_text=(
            format_fit_summary_lines(eta_fit_summary)
            if SHOW_FIT_RESULTS else None
        ),
        style_indices={'Reco / Ref': 0},
        output=OUTPUT_DIR / f'{output_prefix}_full_ratio_{output_suffix}.pdf',
        canvas_name=f'{tag}_full_ratio', **common_plot_options,
    )
    canvases['fb_overlay'] = draw_overlay(
        fb_ratios, x_title='|#eta_{CM}^{dijet}|',
        y_title='Forward / Backward', x_range=fb_range, y_range=FB_RANGE,
        reference_y=1.0, style_indices={'Reco': 0, 'Ref': 1},
        output=OUTPUT_DIR / f'{output_prefix}_fb_overlay_{output_suffix}.pdf',
        canvas_name=f'{tag}_fb_overlay', **common_plot_options,
    )
    canvases['fb_ratio'] = draw_overlay(
        fb_comparisons, x_title='|#eta_{CM}^{dijet}|',
        y_title='(F/B)_{Reco} / (F/B)_{Ref}', x_range=fb_range,
        y_range=FB_DOUBLE_RATIO_RANGE, reference_y=1.0,
        overlay_functions=fb_fits,
        overlay_text=(
            format_fit_summary_lines(fb_fit_summary)
            if SHOW_FIT_RESULTS else None
        ),
        style_indices={'(F/B) Reco / Ref': 0},
        output=OUTPUT_DIR / f'{output_prefix}_fb_ratio_{output_suffix}.pdf',
        canvas_name=f'{tag}_fb_ratio', **common_plot_options,
    )

    suffix = 'smoothed' if APPLY_SYSTEMATIC_SMOOTHING else 'nonsmoothed'
    eta_systematic_csv = write_systematic_csv(
        eta_systematic,
        OUTPUT_DIR / f'{output_prefix}_systematic_{suffix}_full_relative_{output_suffix}.csv',
        evaluation_range=(-ETA_CUT, ETA_CUT),
    )
    fb_systematic_csv = write_systematic_csv(
        fb_systematic,
        OUTPUT_DIR / f'{output_prefix}_systematic_{suffix}_fb_relative_{output_suffix}.csv',
        evaluation_range=(0.0, ETA_CUT),
    )
    canvases['eta_syst'] = draw_overlay(
        {'Pointing resolution': eta_percent},
        x_title='#eta_{CM}^{dijet}', y_title='Pointing Rel. Syst. Uncrt. (%)',
        x_range=eta_range, y_range=SYSTEMATIC_Y_RANGE, show_legend=False,
        output=OUTPUT_DIR / f'{output_prefix}_systematic_{suffix}_full_{output_suffix}.pdf',
        canvas_name=f'{tag}_full_syst', **common_plot_options,
    )
    canvases['fb_syst'] = draw_overlay(
        {'Pointing resolution': fb_percent},
        x_title='|#eta_{CM}^{dijet}|', y_title='Pointing Rel. Syst. Uncrt. (%)',
        x_range=fb_range, y_range=SYSTEMATIC_Y_RANGE, show_legend=False,
        output=OUTPUT_DIR / f'{output_prefix}_systematic_{suffix}_fb_{output_suffix}.pdf',
        canvas_name=f'{tag}_fb_syst', **common_plot_options,
    )
    canvases['eta_band'] = draw_ratio_with_systematic_band(
        eta_ratios, eta_systematic, eta_fits, x_range=eta_range,
        y_range=FULL_RATIO_RANGE, x_title='#eta_{CM}^{dijet}',
        y_title='Reco / Ref', annotations=annotations,
        output=OUTPUT_DIR / f'{output_prefix}_full_ratio_band_{output_suffix}.pdf',
        canvas_name=f'{tag}_full_band',
    )
    canvases['fb_band'] = draw_ratio_with_systematic_band(
        fb_comparisons, fb_systematic, fb_fits, x_range=fb_range,
        y_range=FB_DOUBLE_RATIO_RANGE, x_title='|#eta_{CM}^{dijet}|',
        y_title='(F/B)_{Reco} / (F/B)_{Ref}', annotations=annotations,
        output=OUTPUT_DIR / f'{output_prefix}_fb_ratio_band_{output_suffix}.pdf',
        canvas_name=f'{tag}_fb_band',
    )

    pointing_results[ptave_range] = {
        'raw_eta': raw_eta, 'eta_shapes': eta_shapes,
        'selected_keys': dict(HISTOGRAM_KEYS),
        'forward_backward': fb_ratios, 'eta_ratios': eta_ratios,
        'fb_ratios': fb_comparisons, 'eta_fits': eta_fits,
        'fb_fits': fb_fits, 'eta_fit_summary': eta_fit_summary,
        'fb_fit_summary': fb_fit_summary,
        'eta_systematic_unsmoothed': eta_systematic_unsmoothed,
        'fb_systematic_unsmoothed': fb_systematic_unsmoothed,
        'eta_systematic': eta_systematic,
        'fb_systematic': fb_systematic,
        'eta_systematic_csv': eta_systematic_csv,
        'fb_systematic_csv': fb_systematic_csv,
        'canvases': canvases,
    }
    print(ptave_range, eta_fit_summary, fb_fit_summary)
    for canvas in canvases.values():
        display(canvas)

## Lab-frame pointing-resolution estimator
The Lab estimator uses its own eta-cut index and full distributions only. No Lab Forward/Backward ratios are constructed.

In [ ]:
lab_pointing_results = {}
lab_eta_range = (-LAB_ETA_CUT - 0.1, LAB_ETA_CUT + 0.1)

for ptave_range in PTAVE_BINS:
    low, high = ptave_range
    ptave_tag = f'{low:g}_{high:g}'.replace('.', 'p')
    tag = (
        f'{GENERATOR}_{DIRECTION}_pointing_etaLab_'
        f'{round(10 * LAB_ETA_CUT):g}_ptave_{ptave_tag}'
    )
    output_prefix = f'{GENERATOR}_{DIRECTION}_pointing'
    output_suffix = (
        f'{OUTPUT_CONFIGURATION_TAG}_etaLab_{round(10 * LAB_ETA_CUT):g}'
        f'_ptave_{ptave_tag}'
    )

    raw_eta = {
        label: project_data_histogram(
            INPUT_FILE, key, 'eta', ptave_range,
            name=f'h_{tag}_{label.lower()}_raw',
            rebin=REBIN_ETA, normalization='none',
        )
        for label, key in LAB_HISTOGRAM_KEYS.items()
    }
    eta_shapes = {
        label: normalize_histogram(histogram, 'integral')
        for label, histogram in raw_eta.items()
    }
    eta_ratios = {
        'Reco / Ref': ratio_to_nominal(
            eta_shapes['Reco'], eta_shapes['Ref'],
            name=f'h_{tag}_reco_ref', option=FULL_COMPARISON_RATIO_OPTION,
        )
    }
    eta_fits, eta_fit_summary = fit_histogram_variations(
        eta_ratios, formula=FULL_FIT_FUNCTION,
        fit_range=(-LAB_ETA_CUT, LAB_ETA_CUT),
        name_prefix=f'f_{tag}_full', fit_options=EFFECTIVE_FIT_OPTIONS,
        initial_values=FULL_FIT_INITIAL_VALUES,
    )
    use_fit = SYSTEMATIC_EXTRACTION == 'fit'
    eta_systematic_unsmoothed = calculate_one_sided_systematic(
        eta_ratios['Reco / Ref'], name=f'h_{tag}_full_syst',
        variation_function=eta_fits['Reco / Ref'] if use_fit else None,
        evaluation_range=(-LAB_ETA_CUT, LAB_ETA_CUT),
    )
    eta_systematic = (
        smooth_systematic_running_max(
            eta_systematic_unsmoothed,
            name=f'{eta_systematic_unsmoothed.GetName()}_smoothed',
            evaluation_range=(-LAB_ETA_CUT, LAB_ETA_CUT),
            smoothing_origin=0.0,
        )
        if APPLY_SYSTEMATIC_SMOOTHING else eta_systematic_unsmoothed
    )
    eta_percent = eta_systematic.Clone(f'{eta_systematic.GetName()}_percent')
    eta_percent.SetDirectory(0)
    eta_percent.Scale(100.0)
    annotations = (
        GENERATOR.capitalize(), DIRECTION,
        f'{low:g} < p_{{T}}^{{ave,reco}} < {high:g} GeV',
        f'|#eta_{{Lab}}^{{jet}}| < {LAB_ETA_CUT:g}',
        DIJET_DELTA_PHI_SELECTION_LABEL,
    )
    common_plot_options = dict(
        title='', annotations=annotations, grid=DRAW_GRID,
        style=STYLE, save_png=SAVE_PNG,
    )
    suffix = 'smoothed' if APPLY_SYSTEMATIC_SMOOTHING else 'nonsmoothed'
    eta_systematic_csv = write_systematic_csv(
        eta_systematic,
        OUTPUT_DIR / f'{output_prefix}_systematic_{suffix}_full_relative_{output_suffix}.csv',
        evaluation_range=(-LAB_ETA_CUT, LAB_ETA_CUT),
    )
    canvases = {
        'eta_overlay': draw_overlay(
            eta_shapes, x_title='#eta_{Lab}^{dijet}',
            y_title='1/N dN/d#eta_{Lab}^{dijet}', x_range=lab_eta_range,
            style_indices={'Reco': 0, 'Ref': 1},
            output=OUTPUT_DIR / f'{output_prefix}_full_overlay_{output_suffix}.pdf',
            canvas_name=f'{tag}_full_overlay', **common_plot_options,
        ),
        'eta_ratio': draw_overlay(
            eta_ratios, x_title='#eta_{Lab}^{dijet}', y_title='Reco / Ref',
            x_range=lab_eta_range, y_range=FULL_RATIO_RANGE, reference_y=1.0,
            overlay_functions=eta_fits,
            overlay_text=(format_fit_summary_lines(eta_fit_summary) if SHOW_FIT_RESULTS else None),
            style_indices={'Reco / Ref': 0},
            output=OUTPUT_DIR / f'{output_prefix}_full_ratio_{output_suffix}.pdf',
            canvas_name=f'{tag}_full_ratio', **common_plot_options,
        ),
        'eta_syst': draw_overlay(
            {'Pointing resolution': eta_percent},
            x_title='#eta_{Lab}^{dijet}',
            y_title='Pointing Rel. Syst. Uncrt. (%)',
            x_range=lab_eta_range, y_range=SYSTEMATIC_Y_RANGE,
            show_legend=False,
            output=OUTPUT_DIR / f'{output_prefix}_systematic_{suffix}_full_{output_suffix}.pdf',
            canvas_name=f'{tag}_full_syst', **common_plot_options,
        ),
    }
    canvases['eta_band'] = draw_ratio_with_systematic_band(
        eta_ratios, eta_systematic, eta_fits, x_range=lab_eta_range,
        y_range=FULL_RATIO_RANGE, x_title='#eta_{Lab}^{dijet}',
        y_title='Reco / Ref', annotations=annotations,
        output=OUTPUT_DIR / f'{output_prefix}_full_ratio_band_{output_suffix}.pdf',
        canvas_name=f'{tag}_full_band',
    )
    lab_pointing_results[ptave_range] = {
        'raw_eta': raw_eta, 'eta_shapes': eta_shapes,
        'selected_keys': dict(LAB_HISTOGRAM_KEYS),
        'eta_ratios': eta_ratios, 'eta_fits': eta_fits,
        'eta_fit_summary': eta_fit_summary,
        'eta_systematic_unsmoothed': eta_systematic_unsmoothed,
        'eta_systematic': eta_systematic,
        'eta_systematic_csv': eta_systematic_csv, 'canvases': canvases,
    }
    print('Lab', ptave_range, eta_fit_summary)
    for canvas in canvases.values():
        display(canvas)

In [ ]:
for ptave_range, result in pointing_results.items():
    normalized_integrals = {
        label: histogram.Integral()
        for label, histogram in result['eta_shapes'].items()
    }
    assert all(
        abs(integral - 1.0) < 1e-9
        for integral in normalized_integrals.values()
    ), normalized_integrals
    for histogram in result['forward_backward'].values():
        assert not histogram.GetDirectory()
    print(
        ptave_range, 'normalized integrals', normalized_integrals,
        'selected keys', result['selected_keys'],
    )

for ptave_range, result in lab_pointing_results.items():
    normalized_integrals = {
        label: histogram.Integral()
        for label, histogram in result['eta_shapes'].items()
    }
    assert all(
        abs(integral - 1.0) < 1e-9
        for integral in normalized_integrals.values()
    ), normalized_integrals
    print(
        'Lab', ptave_range, 'normalized integrals', normalized_integrals,
        'selected keys', result['selected_keys'],
    )